Melgren
=======

### Import

In [1]:
from chess import Board, pgn 
from ridoc import matrix_from_board
import torch
from violet import ChessModel
import numpy as np

## Prediction

### Data preparation

In [2]:
def prepare_data(board: Board):
    matrix = matrix_from_board(board)
    position_tensor = torch.tensor(matrix, dtype=np.float32).unsqueeze(0)
    return position_tensor

### Load the model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "1_0_50"
model = ChessModel()
model.load_state_dict(torch.load(f"../lib/models/{model_name}.pth"))
model.eval()

def predict_move(board: Board):
    position = prepare_data(board)

    with torch.no_grad():
        logits = model(position)
    
    probabilities = torch.softmax(logits).cpu().numpy()
    legal_moves = list(board.legal_moves)
    legal_moves_array = np.full((64, 64), False, dtype=np.bool)
    for move in legal_moves:
        legal_moves_array[move.from_square, move.to_square] = True
    
    probabilities = torch.tensor(np.ma.fix_invalid(probabilities, mask=legal_moves_array, fill_value=0))
    probabilities = torch.softmax(probabilities).cpu().numpy()
    move_index = np.argmax(probabilities)
    print(move_index)

predict_move(Board())



Using device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '..\\lib\\models\\1_0_50.pth'